In [1]:
import pandas as pd
from sqlalchemy import create_engine

df = pd.read_csv("cleaned_superstore.csv")

df['Order Date'] = pd.to_datetime(df['Order Date'], format='mixed').dt.strftime('%Y-%m-%d')
df['Ship Date'] = pd.to_datetime(df['Ship Date'], format='mixed').dt.strftime('%Y-%m-%d')

df.columns = [c.replace(' ', '_').replace('-', '_') for c in df.columns]

engine = create_engine('sqlite:///superstore.db')
df.to_sql('orders', engine, if_exists='replace', index=False)

print("Success! 'superstore.db' created with standardized dates and clean columns.")


Success! 'superstore.db' created with standardized dates and clean columns.


In [2]:
q1 = """
SELECT Product_Name, ROUND(SUM(Sales), 2) AS Total_Sales
FROM orders
GROUP BY Product_Name
ORDER BY Total_Sales DESC
LIMIT 5;
"""
pd.read_sql(q1, engine)


,Product_Name,Total_Sales
0,Canon imageCLASS 2200 Advanced Copier,61599.82
1,Fellowes PB500 Electric Punch Plastic Comb Bin...,27453.38
2,Cisco TelePresence System EX90 Videoconferenci...,22638.48
3,HON 5400 Series Task Chairs for Big and Tall,21870.58
4,GBC DocuBind TL300 Electric Binding System,19823.48


In [3]:
q2 = """
SELECT SUBSTR(Order_Date, 1, 7) AS Year_Month,
       ROUND(SUM(Sales), 2) AS Monthly_Sales,
       ROUND(SUM(Profit), 2) AS Monthly_Profit
FROM orders
GROUP BY Year_Month
ORDER BY Year_Month;
"""
pd.read_sql(q2, engine)


,Year_Month,Monthly_Sales,Monthly_Profit
0,2014-01,14236.90,2450.19
1,2014-02,4519.89,862.31
2,2014-03,55691.01,498.73
3,2014-04,28295.35,3488.84
4,2014-05,23648.29,2738.71
5,2014-06,34595.13,4976.52
6,2014-07,33946.39,-841.48
7,2014-08,27909.47,5318.10
8,2014-09,81777.35,8328.10
9,2014-10,31453.39,3448.26


In [4]:
q3 = """
WITH CustomerSpend AS (
    SELECT Customer_ID, Customer_Name, SUM(Sales) AS Total_Spend
    FROM orders
    GROUP BY Customer_ID, Customer_Name
)
SELECT Customer_Name,
       ROUND(Total_Spend, 2) AS Total_Spend,
       CASE 
           WHEN Total_Spend >= 5000 THEN 'VIP'
           WHEN Total_Spend >= 2000 THEN 'Regular'
           ELSE 'Low Spender'
       END AS Customer_Segment
FROM CustomerSpend
ORDER BY Total_Spend DESC
LIMIT 10;
"""
pd.read_sql(q3, engine)


,Customer_Name,Total_Spend,Customer_Segment
0,Sean Miller,25043.05,VIP
1,Tamara Chand,19052.22,VIP
2,Raymond Buch,15117.34,VIP
3,Tom Ashbrook,14595.62,VIP
4,Adrian Barton,14473.57,VIP
5,Ken Lonsdale,14175.23,VIP
6,Sanjit Chand,14142.33,VIP
7,Hunter Lopez,12873.30,VIP
8,Sanjit Engle,12209.44,VIP
9,Christopher Conant,12129.07,VIP


In [5]:
q4 = """
WITH SubCatSales AS (
    SELECT Category, Sub_Category, ROUND(SUM(Sales), 2) AS Total_Sales
    FROM orders
    GROUP BY Category, Sub_Category
)
SELECT * FROM (
    SELECT Category, Sub_Category, Total_Sales,
           RANK() OVER (PARTITION BY Category ORDER BY Total_Sales DESC) AS Rank_In_Category
    FROM SubCatSales
)
WHERE Rank_In_Category <= 3;
"""
pd.read_sql(q4, engine)


,Category,Sub_Category,Total_Sales,Rank_In_Category
0,Furniture,Chairs,328449.10,1
1,Furniture,Tables,206965.53,2
2,Furniture,Bookcases,114880.00,3
3,Office Supplies,Storage,223843.61,1
4,Office Supplies,Binders,203412.73,2
5,Office Supplies,Appliances,107532.16,3
6,Technology,Phones,330007.05,1
7,Technology,Machines,189238.63,2
8,Technology,Accessories,167380.32,3


In [6]:
q5 = """
WITH YearlySales AS (
    SELECT SUBSTR(Order_Date, 1, 4) AS Order_Year,
           ROUND(SUM(Sales), 2) AS Annual_Sales
    FROM orders
    GROUP BY Order_Year
)
SELECT Order_Year,
       Annual_Sales,
       LAG(Annual_Sales) OVER (ORDER BY Order_Year) AS Previous_Year_Sales
FROM YearlySales;
"""
pd.read_sql(q5, engine)


,Order_Year,Annual_Sales,Previous_Year_Sales
0,2014,484247.50,NaN
1,2015,470532.51,484247.50
2,2016,609205.60,470532.51
3,2017,733215.26,609205.60


In [7]:
q6 = """
SELECT Region, Category,
       ROUND(SUM(Sales), 2) AS Total_Sales,
       ROUND(SUM(Profit), 2) AS Total_Profit
FROM orders
GROUP BY Region, Category
HAVING SUM(Profit) > 10000
ORDER BY Total_Profit DESC;
"""
pd.read_sql(q6, engine)


,Region,Category,Total_Sales,Total_Profit
0,West,Office Supplies,220853.25,52609.85
1,East,Technology,264973.98,47462.04
2,West,Technology,251991.83,44303.65
3,East,Office Supplies,205516.05,41014.58
4,Central,Technology,170416.31,33697.43
5,South,Technology,148771.91,19991.83
6,South,Office Supplies,125651.31,19986.39
7,West,Furniture,252612.74,11504.95


In [8]:
q7 = """
SELECT Sub_Category,
       ROUND(SUM(Sales), 2) AS Total_Sales,
       ROUND(SUM(Profit), 2) AS Total_Loss
FROM orders
GROUP BY Sub_Category
HAVING SUM(Profit) < 0
ORDER BY Total_Loss ASC;
"""
pd.read_sql(q7, engine)


,Sub_Category,Total_Sales,Total_Loss
0,Tables,206965.53,-17725.48
1,Bookcases,114880.00,-3472.56
2,Supplies,46673.54,-1189.10


In [9]:
q8 = """
SELECT Segment,
       COUNT(DISTINCT Order_ID) AS Total_Orders,
       ROUND(SUM(Sales), 2) AS Total_Sales,
       ROUND(SUM(Profit), 2) AS Total_Profit
FROM orders
GROUP BY Segment
ORDER BY Total_Profit DESC;
"""
pd.read_sql(q8, engine)


,Segment,Total_Orders,Total_Sales,Total_Profit
0,Consumer,2586,1161401.34,134119.21
1,Corporate,1514,706146.37,91979.13
2,Home Office,909,429653.15,60298.68


In [10]:
q9 = """
SELECT Ship_Mode,
       COUNT(DISTINCT Order_ID) AS Total_Orders,
       ROUND(AVG(Discount) * 100, 1) AS Avg_Discount_Pct
FROM orders
GROUP BY Ship_Mode
ORDER BY Total_Orders DESC;
"""
pd.read_sql(q9, engine)


,Ship_Mode,Total_Orders,Avg_Discount_Pct
0,Standard Class,2994,16.0
1,Second Class,964,13.9
2,First Class,787,16.5
3,Same Day,264,15.2


In [11]:
from sqlalchemy import text

with engine.connect() as conn:
    conn.execute(text("""
        CREATE VIEW IF NOT EXISTS state_performance AS
        SELECT Region, State,
               COUNT(DISTINCT Order_ID) AS Orders,
               ROUND(SUM(Sales), 2) AS Total_discount,
               ROUND(SUM(Profit), 2) AS Total_Profit
        FROM orders
        GROUP BY Region, State;
    """))
    conn.commit()

pd.read_sql("SELECT * FROM state_performance ORDER BY Total_Profit DESC LIMIT 10;", engine)


,Region,State,Orders,Total_Sales,Total_Profit
0,West,California,1021,457687.63,76381.39
1,East,New York,562,310876.27,74038.55
2,West,Washington,256,138641.27,33402.65
3,Central,Michigan,117,76269.61,24463.19
4,South,Virginia,115,70636.72,18597.95
5,Central,Indiana,73,53555.36,18382.94
6,South,Georgia,91,49095.84,16250.04
7,South,Kentucky,61,36591.75,11199.70
8,Central,Minnesota,44,29863.15,10823.19
9,East,Delaware,44,27451.07,9977.37


In [12]:
query_window = """
WITH YearlySales AS (
    SELECT SUBSTR(Order_Date, 1, 4) AS Year, ROUND(SUM(Sales), 2) AS Annual_Sales
    FROM orders
    GROUP BY Year
)
SELECT Year, Annual_Sales,
       LAG(Annual_Sales) OVER (ORDER BY Year) AS Prev_Year_Sales,
       LEAD(Annual_Sales) OVER (ORDER BY Year) AS Next_Year_Sales,
       RANK() OVER (ORDER BY Annual_Sales DESC) AS Sales_Rank
FROM YearlySales;
"""
pd.read_sql(query_window, engine)


,Year,Annual_Sales,Prev_Year_Sales,Next_Year_Sales,Sales_Rank
0,2014,484247.50,NaN,470532.51,3
1,2015,470532.51,484247.50,609205.60,4
2,2016,609205.60,470532.51,733215.26,2
3,2017,733215.26,609205.60,NaN,1


In [6]:
import pandas as pd
from sqlalchemy import create_engine

df = pd.read_csv("cleaned_superstore.csv")
df['Order Date'] = pd.to_datetime(df['Order Date'], format='mixed').dt.strftime('%Y-%m-%d')
df['Ship Date'] = pd.to_datetime(df['Ship Date'], format='mixed').dt.strftime('%Y-%m-%d')
df.columns = [c.replace(' ', '_').replace('-', '_') for c in df.columns]

engine = create_engine('sqlite:///superstore.db')

customers_df = df[['Customer_ID', 'Customer_Name', 'Segment', 'Region']].drop_duplicates(subset=['Customer_ID'])
customers_df.to_sql('customers', engine, if_exists='replace', index=False)

df.to_sql('orders', engine, if_exists='replace', index=False)

print("Database ready with 'orders' and 'customers' tables!")


Database ready with 'orders' and 'customers' tables!


In [7]:
%%writefile superstore_queries.sql
-- Q1: Top 5 products by sales
SELECT Product_Name, ROUND(SUM(Sales), 2) AS Sales FROM orders GROUP BY Product_Name ORDER BY Sales DESC LIMIT 5;

-- Q2: Monthly sales trend
SELECT SUBSTR(Order_Date, 1, 7) AS Month, ROUND(SUM(Sales), 2) AS Monthly_Sales FROM orders GROUP BY Month ORDER BY Month;

-- Q3: Customer segmentation by spend
WITH Spend AS (SELECT Customer_ID, SUM(Sales) AS Total FROM orders GROUP BY Customer_ID)
SELECT CASE WHEN Total >= 5000 THEN 'VIP' WHEN Total >= 2000 THEN 'Regular' ELSE 'Low' END AS Tier,
       COUNT(*) AS Customer_Count, ROUND(AVG(Total), 2) AS Avg_Spend
FROM Spend GROUP BY Tier;

-- Q4: Sales and profit by Region (Using JOIN)
SELECT c.Region, ROUND(SUM(o.Sales), 2) AS Total_Sales, ROUND(SUM(o.Profit), 2) AS Total_Profit
FROM orders o JOIN customers c ON o.Customer_ID = c.Customer_ID
GROUP BY c.Region ORDER BY Total_Sales DESC;

-- Q5: Sub-categories with negative profit (Using HAVING)
SELECT Sub_Category, ROUND(SUM(Profit), 2) AS Total_Loss FROM orders GROUP BY Sub_Category HAVING SUM(Profit) < 0 ORDER BY Total_Loss;

-- Q6: Top 3 products per Category (Using ROW_NUMBER)
SELECT * FROM (
    SELECT Category, Product_Name, ROUND(SUM(Sales), 2) AS Sales,
           ROW_NUMBER() OVER (PARTITION BY Category ORDER BY SUM(Sales) DESC) AS rn
    FROM orders GROUP BY Category, Product_Name
) WHERE rn <= 3;

-- Q7: Most profitable Customer Segment (Using JOIN)
SELECT c.Segment, COUNT(DISTINCT o.Order_ID) AS Orders, ROUND(SUM(o.Profit), 2) AS Profit
FROM orders o JOIN customers c ON o.Customer_ID = c.Customer_ID
GROUP BY c.Segment ORDER BY Profit DESC;

-- Q8: Ship Mode usage and average discount
SELECT Ship_Mode, COUNT(DISTINCT Order_ID) AS Total_Orders, ROUND(AVG(Discount), 2) AS Avg_Discount FROM orders GROUP BY Ship_Mode;

-- Q9: Top 5 States by total sales
SELECT State, ROUND(SUM(Sales), 2) AS Total_Sales FROM orders GROUP BY State ORDER BY Total_Sales DESC LIMIT 5;

-- Q10: High-discount orders (> 50%) impact on profit (Using WHERE)
SELECT Category, COUNT(*) AS High_Discount_Orders, ROUND(SUM(Profit), 2) AS Net_Profit FROM orders WHERE Discount > 0.5 GROUP BY Category;


Writing superstore_queries.sql


In [8]:
%%writefile README.md

Analyzed the Superstore retail dataset using SQLite and Python (`pandas` + `SQLAlchemy`) to extract business insights across product performance, customer tiers, regional profitability, and discounting impact.

- `task2_sql_analysis.ipynb`: Jupyter Notebook executing 10 business queries with outputs and insights.
- `superstore_queries.sql`: Standalone SQL script containing all queries.
- `db_utils.py`: Reusable Python database connection and query execution module.

- **Top Revenue Concentration:** *Canon imageCLASS 2200 Advanced Copier* is the highest revenue generator, with **Technology** leading all categories in sales and overall net profit.
- **Sub-Category Loss Leaks:** Filtering via `GROUP BY` and `HAVING` confirms that **Tables**, **Bookcases**, and **Supplies** operate at a net financial loss despite steady order volumes.
- **Heavy Discount Threshold:** Orders discounted above **50%** consistently generate steep net losses across categories, highlighting the need for stricter discount caps.
- **Customer Value Tiering:** CTE-based segmentation (`WITH` + `CASE WHEN`) shows that **VIP** and **Regular** spenders drive a disproportionate share of revenue, with the **Consumer** segment leading total order volume and profit.
- **Regional & Seasonal Growth:** Multi-table `JOIN` analyses show that **West** and **East** regions anchor company profitability, while monthly trends peak sharply in **November** and **December**.


Writing README.md
